# Benchmark summary tables

Scores all methods and draws one table per cohort plus a combined table.

- Metrics are averaged over samples, then min–max scaled per cohort across non-control methods.
- MECR is size-adjusted per cohort and inverted. Cell density is log-scaled.
- The combined table averages scaled metrics across cohorts.
- The total score is the mean of the available categories.
- Methods marked * are missing in some cohorts (Merlin: aging only). Their combined values average the available cohorts.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import font_manager

sys.path.insert(0, str(Path.cwd().parents[0]))
from cellseg_benchmark._constants import BASE_PATH, celltype_groups, main_methods, merged_celltypes
from cellseg_benchmark.metrics import clean_method_name
from cellseg_benchmark.metrics.benchmark_table import (
    build_benchmark_tables,
    categories,
    exclude,
    metrics,
    plot_benchmark_table,
)

cohorts = ["aging", "ABCAtlas", "VizgenMouseBrain"]
out_dir = Path(BASE_PATH) / "metrics" / "merged" / "benchmark_tables"

## Build tables

In [ ]:
tables, pooled_score = build_benchmark_tables(cohorts)

## Plot tables

In [ ]:
for name, table in tables.items():
    plot_benchmark_table(table[table.index.isin(main_methods)], out_dir / "all" / f"{name}_main")
    plot_benchmark_table(table, out_dir / "all" / f"{name}_full")

df_expr = tables["combined"].assign(total_score=lambda df: df.expr_score).sort_values("total_score", ascending=False)
plot_benchmark_table(df_expr, out_dir / "all" / "combined_expression", score_label="Expr. score")

## Cell-type tables

Same metrics per cell type in `celltype_groups` (neuron subtypes merged, as in the marker list). Metrics are scaled within the cell type, so scores rank methods within a cell type and are not comparable across cell types. MECR uses the size fit of the all-cells table of the same cohort. Silhouette is undefined within one cell type, so totals average marker, Ficture and Z.

In [ ]:
type_tables = {t: build_benchmark_tables(cohorts, t)[0] for t in celltype_groups}
for group, group_tables in type_tables.items():
    for name, table in group_tables.items():
        plot_benchmark_table(table[table.index.isin(main_methods)], out_dir / group / f"{name}_main")
        plot_benchmark_table(table, out_dir / group / f"{name}_full")

## Sensitivity checks

In [ ]:
real = tables["combined"].query("~is_control")
cohort_scores = pd.DataFrame({c: tables[c].total_score for c in cohorts}).reindex(real.index)
mean_rank = cohort_scores.rank(ascending=False).mean(axis=1)

print("shared range and MECR fit:", real.total_score.corr(pooled_score[real.index], method="spearman"))
print("mean rank across cohorts:", real.total_score.corr(-mean_rank, method="spearman"))

## Missing values

In [ ]:
for name, table in tables.items():
    missing = table[metrics].isna()
    missing["ficture_f1_score"] &= ~table.na_ficture
    missing["Ovrlpy_stats_mean_integrity"] &= ~table.na_z
    print(name, {metric: missing.index[missing[metric]].tolist() for metric in metrics if missing[metric].any()})

## Cell-type comparison

All main methods. 3D methods lack Ficture F1 and Z integrity, so as in Fig. 1 their totals average the available categories; the check below compares totals with the marker score, which all methods share. The matrix shows within-cell-type scores (colour) and ranks (numbers). The panel shows metrics per cell type on the Fig. 1 scale, uncapped: 0 and 1 are the worst and best method on all cells (shaded; pooled across cohorts), higher is better. It shows which cell types are hard for all methods. Dashed line: all-cells median. MECR adjusted with the all-cells transcripts-per-cell fit, then inverted. Neuron Marker F1 is low because pan-neuronal genes fail the marker specificity filter; with canonical Glut/Gaba markers it is the highest of all types (aging).

In [ ]:
font_manager.fontManager.addfont(Path(BASE_PATH) / "misc" / "Arial.ttf")
plt.rcParams.update({
    "font.family": "Arial", "font.size": 7, "axes.linewidth": 0.6, "xtick.major.width": 0.6,
    "xtick.major.size": 2.5, "ytick.major.size": 0, "svg.fonttype": "none", "pdf.fonttype": 42,
})
combined = tables["combined"]
methods = [m for m in combined.index if m in main_methods]
scores = pd.DataFrame({"All cells": combined.total_score} | {t: type_tables[t]["combined"].total_score for t in celltype_groups}).loc[methods]
scores.index = [f"{m}*" if combined.partial[m] else m for m in methods]
pd.concat({"all": combined} | {t: type_tables[t]["combined"] for t in celltype_groups}, names=["group", "method"])[
    ["total_score", *categories, "is_control"]].to_csv(out_dir / "celltype_scores_scaled.csv")

fig, ax = plt.subplots(figsize=(120 / 25.4, len(methods) * 3.3 / 25.4))
sns.heatmap(scores, cmap="YlOrBr", vmin=0, vmax=1, xticklabels=True, yticklabels=True, annot=scores.rank(ascending=False), fmt=".0f", annot_kws={"size": 6},
            linewidths=0.5, linecolor="white", cbar_kws={"label": "Score (scaled within cell type)", "shrink": 0.6}, ax=ax)
ax.set(xlabel="", ylabel="")
ax.tick_params(length=0)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
for ext in ["pdf", "svg", "png"]:
    fig.savefig(out_dir / f"celltype_matrix.{ext}", bbox_inches="tight", dpi=600, transparent=True)

In [ ]:
pd.Series({
    t: type_tables[t]["combined"].loc[methods].query("~is_control").pipe(lambda d: d.total_score.corr(d.marker, method="spearman"))
    for t in celltype_groups
}).round(2)

In [ ]:
raw = {
    t: pd.concat([pd.read_csv(Path(BASE_PATH) / "metrics" / c / f / f"{t}.csv", index_col=0) for c in cohorts]).query("sample not in ['all', 'all_samples']")
    for t, f in [("ficture_f1", "ficture"), ("marker_f1_score_cell_type_revised", "marker_gene_metrics"), ("general_stats", "general_stats"),
                 ("MECR_score_all", "marker_gene_metrics"), ("MECR_score_groups", "marker_gene_metrics")]
}
weighted = ["Ovrlpy_stats_mean_integrity", "n_counts", "elongation", "volume_final"]
rows = []
for group, types in ({"All cells": None} | celltype_groups).items():
    fic = raw["ficture_f1"][~raw["ficture_f1"].cell_type.str.startswith("f1_") & (raw["ficture_f1"].cell_type != "BAMs")]
    mk, gs = raw["marker_f1_score_cell_type_revised"], raw["general_stats"]
    if types:
        fic = fic[fic.cell_type.isin(types)]
        mk = mk[mk.cell_type.isin({merged_celltypes.get(t, t) for t in types})]
        gs = gs[gs.cell_type_revised.isin(types)]
        mecr = raw["MECR_score_groups"].query("group == @group")
    else:
        gs = gs[gs.cell_type_revised == "all"]
        mecr = raw["MECR_score_all"]
    gs = gs[weighted].mul(gs.n_cells, axis=0).assign(n_cells=gs.n_cells, method=gs.method).groupby("method").sum(min_count=1)
    gs = gs[weighted].div(gs.n_cells, axis=0)
    mk = mk.groupby(["sample", "method", "cell_type"]).f1_score.mean().groupby(["sample", "method"]).mean().groupby("method").mean()
    rows.append(pd.DataFrame({
        "Ficture F1": fic.groupby(["sample", "method"]).f1.mean().groupby("method").mean(),
        "Z integrity": gs.Ovrlpy_stats_mean_integrity,
        "Marker F1": mk,
        "MECR": mecr.query("gene1 == 'all'").groupby("method").MECR.mean(),
        "tpc": gs.n_counts,
        "Elongation": gs.elongation,
        "Volume": gs.volume_final,
    }).assign(group=group))

df = pd.concat(rows).rename_axis("method").reset_index()
df = df[~df.method.str.startswith("Negative_Control") & ~df.method.isin(exclude)]
fit_rows = df[df.group == "All cells"].dropna(subset=["MECR", "tpc"])
df["Adjusted MECR"] = df.MECR - np.polyval(np.polyfit(np.log10(fit_rows.tpc), fit_rows.MECR, 1), np.log10(df.tpc))
panels = ["Ficture F1", "Z integrity", "Marker F1", "Adjusted MECR"]
ref = df.loc[df.group == "All cells", panels]
df[panels] = (df[panels] - ref.min()) / (ref.max() - ref.min())
df["Adjusted MECR"] = 1 - df["Adjusted MECR"]
df = df[df.method.map(clean_method_name).isin(methods)]

In [ ]:
order = ["All cells", *celltype_groups]
fig, axes = plt.subplots(1, 4, figsize=(144 / 25.4, 70 / 25.4), sharey=True, gridspec_kw={"wspace": 0.15})
for ax, col in zip(axes, panels):
    ax.axvspan(0, 1, color="0.93", lw=0, zorder=0)
    ax.axvline(df.loc[df.group == "All cells", col].median(), color="0.75", lw=0.6, ls="--", zorder=0)
    sns.boxplot(df, x=col, y="group", order=order, ax=ax, showfliers=False, width=0.6,
                boxprops={"facecolor": "none", "edgecolor": "black", "linewidth": 0.6},
                whiskerprops={"linewidth": 0.6}, capprops={"linewidth": 0}, medianprops={"color": "black", "linewidth": 1})
    sns.stripplot(df, x=col, y="group", order=order, ax=ax, color="0.55", size=2, jitter=0.18, linewidth=0, zorder=0)
    ax.set_title(col, fontsize=7, pad=4)
    ax.set(xlabel="", ylabel="")
    ax.xaxis.set_major_locator(plt.MaxNLocator(3))
    sns.despine(ax=ax, left=True)
df.to_csv(out_dir / "celltype_metrics_raw.csv", index=False)
for ext in ["pdf", "svg", "png"]:
    fig.savefig(out_dir / f"celltype_difficulty.{ext}", bbox_inches="tight", dpi=600, transparent=True)

Does difficulty follow cell shape? Spearman across cell types (n = 10), using medians over methods. Agreement: how consistently methods rank cell types by elongation.

In [ ]:
med = df.groupby("group")[[*panels, "Elongation", "Volume"]].median().drop("All cells")
rho = df.pivot_table(index="group", columns="method", values="Elongation").drop("All cells").corr("spearman")
print("agreement:", rho.values[np.triu_indices(len(rho), 1)].mean().round(2))
med.corr("spearman").loc[panels, ["Elongation", "Volume"]].round(2)